<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_05_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 05 — The Report

**Paired with lecture block L6 · Part 1**

The last notebook of Part 1. What is marked is not whether your numbers match
anyone else's — it is whether you can say what you measured, what it means, and
where it stops being true.

## How to use this notebook

Run section 0 to pull in what notebooks 01 to 04 saved. Then replace the two
strings in section 2, one for each lecture of the block. Section 3 assembles the whole thing into a markdown file
you submit.

Word limits are enforced by the check in section 3, and they are tight on
purpose. An answer that needs six hundred words has not been understood yet.

## 0 · Load what the other notebooks produced

In [21]:
# files-cell v2 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository on every run, so a file left
# over from an earlier session cannot shadow it. Run this cell first.
import os, sys, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    # on Colab always re-fetch: /content outlives a session and a stale copy
    # would silently shadow an updated one. Locally the repo's own files stay.
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
# a module imported before this cell would keep the previous file's contents
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))


fetched Ex_6_core.py
files ready: Ex_6_core.py


In [22]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.

import Ex_6_core as core
core.keep_outputs()

Google Drive is not available (Error: credential propagation was unsuccessful).
Results will be downloaded to your computer instead. Keep the
files: the notebook that needs them asks for them.


'/content/Ex06_outputs'

In [23]:
import os
import numpy as np
import Ex_6_core as core

paths = {
    "01 · losses":       "nb01_losses.npz",
    "02 · optimisers":   "nb02_optimisers.npz",
    "03 · transfer":     "nb03_transfer.npz",
    "04 · arbitrage":    "nb04_arbitrage.npz",
}

R = {}
core.needed('nb01_losses.npz', 'nb02_optimisers.npz', 'nb03_transfer.npz',
            'nb04_arbitrage.npz')   # on Colab without Drive, asks for the missing files
for label, fname in paths.items():
    full = os.path.join(core.OUTPUT_DIR, fname)
    if os.path.exists(full):
        R[label] = np.load(full, allow_pickle=True)
        print(f"  loaded  {label}")
    else:
        print(f"  MISSING {label}  ({fname}) -- run that notebook first")

if len(R) < len(paths):
    print("\nSome results are missing. The report will have gaps.")

  loaded  01 · losses
  loaded  02 · optimisers
  loaded  03 · transfer
  loaded  04 · arbitrage


---

## 0b · Your personal seed

Every notebook in this exercise set fixes the seed to 0, so the printed
"what you should see" blocks are true on any machine. That is right for
checking your work and wrong for reporting it — with one seed the whole cohort
produces identical numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints in your answers where the questions ask for it. Your
supervisor can regenerate exactly these numbers from your study number alone,
so they are worth getting right and pointless to invent.

In [24]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = core.personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own calibration run on the same load cell.
x_you, y_you = core.calibration_dataset(n=40, seed=SEED)
design = np.stack([x_you, np.ones_like(x_you)], axis=1)
A_YOU, B_YOU = np.linalg.lstsq(design, y_you, rcond=None)[0]

print()
print(f"  your fitted slope     : {A_YOU:.4f}   (true {core.TRUE_SLOPE})")
print(f"  your fitted intercept : {B_YOU:.4f}   (true {core.TRUE_INTERCEPT})")

study number : 20241234
your seed    : 47531

  your fitted slope     : 2.3064   (true 2.4)
  your fitted intercept : 0.9295   (true 0.8)


## 1 · The evidence, gathered

Read these off before writing anything. Several of the questions below are
about numbers you may not have looked at since you produced them.

In [25]:
if "01 · losses" in R:
    d = R["01 · losses"]
    print("NOTEBOOK 01")
    print(f"  least-squares slope/intercept : {d['a_ls']:.4f} / {d['b_ls']:.4f}")
    print(f"  cross entropy, numpy vs torch : {abs(d['ce_numpy']-d['ce_torch']):.2e}")
    print(f"  accuracy   CE / MSE           : {d['acc_ce']:.3f} / {d['acc_mse']:.3f}")
    print(f"  held-out CE, CE / MSE trained : {d['heldout_ce_ce']:.4f} / "
          f"{d['heldout_ce_mse']:.4f}")
    print(f"  line fit under MSE / MAE      : {d['fit_mse'][0]:.3f} / "
          f"{d['fit_mae'][0]:.3f}   (true {core.TRUE_SLOPE})")

if "02 · optimisers" in R:
    d = R["02 · optimisers"]
    print("\nNOTEBOOK 02")
    for n, v in zip(d["final_names"], d["final_losses"]):
        print(f"  {str(n):<22s} {v:.3e}")

if "03 · transfer" in R:
    d = R["03 · transfer"]
    print("\nNOTEBOOK 03")
    print(f"  source model on A / on B : {d['acc_source_A']:.3f} / "
          f"{d['acc_source_B']:.3f}")
    for n, v in zip(d["strategy_names"], d["strategy_acc"]):
        print(f"  {str(n):<28s} {v:.3f}")

if "04 · arbitrage" in R:
    d = R["04 · arbitrage"]
    print("\nNOTEBOOK 04")
    lp = float(d["lp"].mean())
    for name in ("lp", "rl", "imitation"):
        v = float(d[name].mean())
        print(f"  {name:<10s} {v:6.2f} EUR per day   ({100 * v / lp:3.0f} % of the optimum)")
    print(f"  linear program / network : {float(d['lp_ms']):.3f} / {float(d['net_ms']):.3f} ms per day")


NOTEBOOK 01
  least-squares slope/intercept : 2.3954 / 0.7871
  cross entropy, numpy vs torch : 0.00e+00
  accuracy   CE / MSE           : 0.942 / 0.933
  held-out CE, CE / MSE trained : 0.1961 / 0.2082
  line fit under MSE / MAE      : 2.203 / 2.474   (true 2.4)

NOTEBOOK 02
  Adam 2000              2.251e-05
  L-BFGS 60              4.968e-02
  Adam 1000 -> L-BFGS    8.753e-07
  Adam 2000 -> L-BFGS    2.250e-05

NOTEBOOK 03
  source model on A / on B : 0.892 / 0.667
  scratch                      0.793
  linear probe                 0.800
  fine-tune                    0.840
  fine-tune, lr 0.05           0.800
  machine-A model, untouched   0.667

NOTEBOOK 04
  lp          33.15 EUR per day   (100 % of the optimum)
  rl          27.86 EUR per day   ( 84 % of the optimum)
  imitation   31.05 EUR per day   ( 94 % of the optimum)
  linear program / network : 16.587 / 0.069 ms per day


## 2 · Your answers

Replace every string. Keep to the word limits.

In [26]:
# TODO: write your report. Every string below must be replaced.

Q1_L61 = """ In my field, as a control engineer, data shows mostly Gaussian behaviour due to much data coming from sensors. I would choose MSE, which is maximum likelihood under gaussian assumption, supported by the MSE having an estimation error of 0.002% under Gaussion noise assumptions.  If im working with encoders, I would choose Huber, as MAE showed better results when having outliers (had a slope / intercept error of 0.0384 and 0.0086 compared to MSE having an error around 0.2). In terms of optimizer, I would choose a combination of ADAM & L-BSFG as this showed a final training loss of 9.190e-07, with the right handoff.
"""

Q2_L62 = """ Starting from a trained model payed off already when using Machine A's model on Machine B, as the right prediction gained from 0.33 to 0.667, meaning that the body of Model-A fit B to some degree of classifying behaviour. But, fine-tuning with a smaller learning rate showed most satisfactorily with an accuracy of 0.840. Reinforcement learning would beat the linear program whenever the problem is not linear, the model is unknown entirely or there exist not an expert to train from. This would leave only the reward to train on, which in notebook 04 gives an optimum of 84% and a 0.071 ms time to plan the day.
"""

NAME = "Miriam"
GROUP = "MCE3-921"



## 3 · Check, assemble, save

The check counts words and refuses anything left at its default. Fix what it
reports, then run it again.

In [27]:
answers = {
    "1 · L6.1 — the loss and the optimiser": (Q1_L61, 150),
    "2 · L6.2 — transfer and reinforcement": (Q2_L62, 150),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or "(%d words)" % limit in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = [f"# Ex_06 — Training Lab", "",
             f"**{NAME}** · {GROUP}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    report = "\n".join(lines)

    os.makedirs(core.OUTPUT_DIR, exist_ok=True)
    out = os.path.join(core.OUTPUT_DIR, "Ex06_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write(report)
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

wrote /content/Ex06_outputs/Ex06_report.md
213 words total


<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 04

Every notebook in this set ended with two questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 9, each under its question, to the end of the
report you just wrote. The last one is not from a notebook: it is the
question across all of them, and it concludes the report. On Colab every notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on its lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [28]:
# notebook-questions v1 -- your answers from notebooks 01 to 04 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · Where the Loss Comes From ----------------------------
    # 01.1 What are the steps from a noise distribution to the loss you
    # minimise? (-> L6.1 Q1)
    "01.1": """ Firstly you assume the noise, eihter Gaussian, Laplace, Bernoulli or categorical. This is then turned into a likelihood for one data point. These are then all combined, taken the logarithmic and then negate it.
""",
    # 01.2 Which loss would you use if one reading in fifty is a transcription
    # error, and what does it cost if the noise was Gaussian after all? (->
    # L6.1 Q2)
    "01.2": """ I would use MAE (Mean Absolute Error). It assumes Laplace noise, which means that it is believed large errors are rare but can occur. This makes it robust to outliers, like if one reading is faulty. It does not consider it important, and is thereby not affected much by it. If the noise is Gaussian, it wastes information and converges slower compared to MSE.
""",

    # ---- notebook 02 · Adam, then L-BFGS ------------------------------------
    # 02.1 What two running averages does Adam keep, and what does each fix
    # compared with plain gradient descent? (-> L6.1 Q4)
    "02.1": """ The average gradient, and the average squared gradient. The average gradient smooths the direction as momentum does, and the average squared gradient sets the size of the parameter's step. These ensure that the learning rate works for both steep and flat directions of the loss, whereas plain GD moves the same for each parameter.
""",
    # 02.2 Why does Adam followed by L-BFGS reach a lower loss than L-BFGS
    # from a cold start? (-> L6.1 Q4)
    "02.2": """ Because ADAM is good far away from the answer, it converges faster and is reliable until close to the minimum where it overshoots and becomes slower. L-BFGS is the opposite,fast close to minimum but unreliable far away. In tow, you get the best of both worlds.
""",

    # ---- notebook 03 · A Second Machine, and Ten Labels Each ----------------
    # 03.1 The machine-A model scored above chance on machine B before any
    # adaptation: what does that say about which layers transfer? (-> L6.2 Q1)
    "03.1": """ It definitely transferred the body of layers, as the accuracy reached 0.667 and thereby improved from 0.333. The early layers being transferred is visualized in that the data is in a class cluster on the graphs. So the body of Machine A, the features thats generalized, also show some relevance to Machine B. The difficulty in placing the data in the intersections of these classes indicate that the head is not similar enough, as a significant amount of data points are wrongly classified.
""",
    # 03.2 What happened when you fine-tuned at the from-scratch learning
    # rate, and what is that failure called? (-> L6.2 Q2)
    "03.2": """ Then a fenomena called catastrophic forgetting happens. This is were larger steps override the pre-trained attributes of the body in the first updates. This makes it so the model behaves more like a from-scratch model instead.
""",

    # ---- notebook 04 · A Battery That Learns to Trade -----------------------
    # 04.1 What are the state, the action and the reward of the battery, and
    # why is the policy judged on the day's profit? (-> L6.2 Q3)
    "04.1": """ The state are charge, hour, price now and price ahead. The actions are buy, wait or sell. The reward is the revenue minus the ageing for every hour. The policy is judged by the profit of the whole day and not each hour, because when buying low, it would only see the net negative of charging, not the positive of selling it at a higher price later on. There is a delayed reward in this setup, which the policy needs to account for in the decision-making.
""",
    # 04.2 Why did the linear program earn more than the learned policy, and
    # when would reinforcement learning be worth its cost? (-> L6.2 Q4)
    "04.2": """ Because its a linear problem, and the battery has linear equations converning it, it can be solved exactly with linear programming. The learned policy based on the reward only comes close. Reinforcement learning is worth it when you dont have a model to optimize or an expert to copy, and is not worth it in this case.
""",

    # ---- to conclude, across all the notebooks ------------------------------
    # C Across the notebooks: which one choice — the loss, the optimiser or
    # the starting weights — changed your result most, and what number shows
    # it? (-> L6.1 and L6.2, Exercise slide)
    "C": """ It was when utilizing L-BFGS together with ADAM an choosing the correct hand-off (No. of epochs). Looking at the different combinations of ADAM, L-BFGS, ADAM + L-BFGS(1000 EPOCHS) and ADAM + L-BFGS (2000 EPOCHS), the difference in training loss is significant. The lowest being just L-BFGS at 3.805e-02 and the highest being ADAM + L-BFGS(1000 EPOCHS) at 9.190e-07. Also worth mentioning is, when going from utilizing the model for Machine-A to using finetuning. The difference in accuracy raising from 0.667 to 0.840 is a significant increase, when classifying data.
""",
}


In [29]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'Where the Loss Comes From', 'What are the steps from a noise distribution to the loss you minimise?', 'L6.1 Q1'),
    "01.2": ('01', 'Where the Loss Comes From', 'Which loss would you use if one reading in fifty is a transcription error, and what does it cost if the noise was Gaussian after all?', 'L6.1 Q2'),
    "02.1": ('02', 'Adam, then L-BFGS', 'What two running averages does Adam keep, and what does each fix compared with plain gradient descent?', 'L6.1 Q4'),
    "02.2": ('02', 'Adam, then L-BFGS', 'Why does Adam followed by L-BFGS reach a lower loss than L-BFGS from a cold start?', 'L6.1 Q4'),
    "03.1": ('03', 'A Second Machine, and Ten Labels Each', 'The machine-A model scored above chance on machine B before any adaptation: what does that say about which layers transfer?', 'L6.2 Q1'),
    "03.2": ('03', 'A Second Machine, and Ten Labels Each', 'What happened when you fine-tuned at the from-scratch learning rate, and what is that failure called?', 'L6.2 Q2'),
    "04.1": ('04', 'A Battery That Learns to Trade', "What are the state, the action and the reward of the battery, and why is the policy judged on the day's profit?", 'L6.2 Q3'),
    "04.2": ('04', 'A Battery That Learns to Trade', 'Why did the linear program earn more than the learned policy, and when would reinforcement learning be worth its cost?', 'L6.2 Q4'),
    "C": ('C', 'to conclude', 'Across the notebooks: which one choice — the loss, the optimiser or the starting weights — changed your result most, and what number shows it?', 'L6.1 and L6.2, Exercise slide'),
}

report_md = os.path.join(core.OUTPUT_DIR, "Ex06_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex06_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "",
           "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += ["### To conclude" if nb == "C" else f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "",
                answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))


added to /content/Ex06_outputs/Ex06_report.md: 9 of 9 answered


**What you should see.** `added to .../Ex06_report.md: 9 of 9 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [30]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex06_report.md into Ex06_report.pdf, with any figure
# saved as Ex06_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(core.OUTPUT_DIR, "Ex06_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(core.OUTPUT_DIR, "Ex06_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex06_report*.png")
                  + glob.glob(os.path.join(core.OUTPUT_DIR, "Ex06_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

written /content/Ex06_outputs/Ex06_report.pdf 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4 · What Ex_06 was for, and where Part 1 ends

Four claims you can now defend with your own measurements:

* a loss is a noise assumption written down, not a preference;
* an optimiser is chosen for where in the landscape you are, which is why
  Part 2 uses two of them in sequence;
* a trained model transfers to a related problem, and the learning rate decides
  whether it survives the transfer;
* a problem with a known model should be solved, and a network that imitates
  the solver is fast enough for the loop.

**Part 1 ends here.** From L7 the physics enters the loss. The machinery stays:
the same networks, the same Adam then L-BFGS. And a small residual is still not
a correct answer.